# 🎬 Python Motion Design — Showcase & Interatividade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ru4nC/skills-motion-design/blob/main/examples/motion_design_showcase.ipynb)

Este notebook demonstra a criação de **Motion Design de alta performance em Python puro** (Pillow + NumPy + FFmpeg), capaz de renderizar:
1. **Cenas Full-Screen (100% Preenchidas em `.mp4`)** com gradientes e contadores numéricos progressivos.
2. **Overlays Transparentes com Canal Alfa Real (`.webm`)** com osciloscópios harmônicos.
3. **Dossiês Investigativos com Marca-Texto Orgânico (Rough Notation)** com perturbações senoidais.

## ⚙️ 1. Configuração do Ambiente Headless (Colab / Jupyter)

In [1]:
# Instalação das dependências mínimas do sistema e fontes
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-liberation fonts-dejavu-core
!pip install -q pillow numpy

import subprocess
import base64
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import IPython.display as ipd

print("✅ Ambiente configurado com sucesso!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-dejavu-mono.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../fonts-dejavu-mono_2.37-8_all.deb ...
Unpacking fonts-dejavu-mono (2.37-8) ...
Selecting previously unselected package fonts-dejavu-core.
Preparing to unpack .../fonts-dejavu-core_2.37-8_all.deb ...
Unpacking fonts-dejavu-core (2.37-8) ...
Setting up fonts-dejavu-mono (2.37-8) ...
Setting up fonts-dejavu-core (2.37-8) ...
Processing triggers for fontconfig (2.15.0-1.1ubuntu2) ...
✅ Ambiente configurado com sucesso!


## 📺 2. Cena Full-Screen: Infográfico 100% Preenchido (.mp4)

In [7]:
def render_fullscreen_frame(t: float, dur: float = 4.0, w: int = 1280, h: int = 720) -> Image.Image:
    # 1. Gradiente procedural contínuo em NumPy
    t_vals = np.linspace(0, 1, h)[:, None]
    r = (15 * (1 - t_vals) + 8 * t_vals).astype(np.uint8)
    g = (22 * (1 - t_vals) + 10 * t_vals).astype(np.uint8)
    b = (36 * (1 - t_vals) + 14 * t_vals).astype(np.uint8)
    a = np.full((h, 1), 255, dtype=np.uint8)
    arr = np.repeat(np.concatenate([r, g, b, a], axis=1)[:, None, :], w, axis=1)
    img = Image.fromarray(arr, mode="RGBA")
    draw = ImageDraw.Draw(img, "RGBA")

    # Tentar carregar fontes escaláveis do sistema
    try:
        font_title = ImageFont.truetype("LiberationSans-Bold.ttf", 28)
        font_metric = ImageFont.truetype("LiberationSans-Bold.ttf", 64)
    except IOError:
        font_title = ImageFont.load_default()
        font_metric = ImageFont.load_default()

    # Easing easeOutExpo
    enter_t = min(1.0, t / 0.5)
    ease = 1.0 - pow(1.0 - enter_t, 3)
    p_roll = min(1.0, max(0.0, t / (dur * 0.7)))
    ease_roll = 1.0 if p_roll >= 0.999 else (1.0 - pow(2.0, -10.0 * p_roll))

    card_w = int(w * 0.75 * ease)
    card_h = int(h * 0.55)
    x0 = (w - int(w * 0.75)) // 2
    y0 = int(h * 0.22)

    if card_w > 20:
        draw.rounded_rectangle([x0, y0, x0 + card_w, y0 + card_h], radius=12, fill=(17, 24, 39, 235), outline=(245, 158, 11, int(220 * ease)), width=2)
        curr_val = int(384400 * ease_roll)
        draw.text((x0 + 40, y0 + 40), "TELEMETRIA ORBITAL DE MISSÃO", fill=(245, 158, 11, int(255 * ease)), font=font_title)
        draw.text((x0 + 40, y0 + 90), f"{curr_val:,} KM".replace(",", "."), fill=(248, 250, 252, int(255 * ease)), font=font_metric)

        # Espessura da barra aumentada para 30px (y0 + 200 até y0 + 230)
        bar_w = int((card_w - 80) * ease_roll)
        if bar_w > 4:
            draw.rectangle([x0 + 40, y0 + 200, x0 + 40 + bar_w, y0 + 230], fill=(6, 182, 212, 240))

    return img

# Renderização via pipe stdin para MP4 H.264 em ~1.2s
fps = 30
dur = 4.0
output_mp4 = "fullscreen_explainer.mp4"
cmd = [
    "ffmpeg", "-y", "-f", "rawvideo", "-s", "1280x720", "-pix_fmt", "rgba",
    "-r", str(fps), "-i", "pipe:0", "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-preset", "fast", "-crf", "18", output_mp4
]
proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
for f in range(int(fps * dur)):
    frame = render_fullscreen_frame(f / float(fps), dur)
    proc.stdin.write(frame.tobytes("raw", "RGBA"))
proc.stdin.close()
proc.wait()

print(f"✅ Vídeo de tela cheia gerado com novas fontes e espessura: {output_mp4}")
ipd.Video(output_mp4, embed=True, width=640)

/tmp/ipykernel_817/1219106563.py:9: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode="RGBA")


✅ Vídeo de tela cheia gerado com novas fontes e espessura: fullscreen_explainer.mp4


## 🎭 3. Overlay Transparente com Canal Alfa Real (.webm) e Validação Xadrez

In [8]:
def render_transparent_hud(t: float, dur: float = 3.5, w: int = 1280, h: int = 720) -> Image.Image:
    # Buffer totalmente transparente (alpha = 0)
    img = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img, "RGBA")

    # Carregar fonte escalável para aumentar as letras
    try:
        font_hud = ImageFont.truetype("LiberationSans-Bold.ttf", 28)
    except IOError:
        font_hud = ImageFont.load_default()

    enter_t = min(1.0, t / 0.45)
    ease = 1.0 - pow(1.0 - enter_t, 3)

    card_w, card_h = int(w * 0.70), int(h * 0.50)
    x0 = (w - card_w) // 2
    y0 = (h - card_h) // 2

    # Card translúcido com neon glow ciano
    panel_bg = (8, 14, 24, int(210 * ease))
    draw.rounded_rectangle([x0, y0, x0 + card_w, y0 + card_h], radius=10, fill=panel_bg, outline=(6, 182, 212, int(240 * ease)), width=2)
    draw.text((x0 + 32, y0 + 32), "HUD ESPECTRAL // OSCILOSCÓPIO HARMÔNICO", fill=(6, 182, 212, int(255 * ease)), font=font_hud)

    # Osciloscópio senoidal multi-harmônico
    osc_w = card_w - 48
    center_y = y0 + 150
    pts = []
    for i in range(70):
        rel_x = i / 69.0
        px = x0 + 24 + int(rel_x * osc_w)
        env = np.sin(rel_x * np.pi)
        w1 = np.sin(rel_x * 12.0 + t * 5.0) * 26.0
        w2 = np.sin(rel_x * 24.0 - t * 7.5) * 12.0
        py = center_y + int((w1 + w2) * env * ease)
        pts.append((px, py))

    for j in range(len(pts) - 1):
        draw.line([pts[j], pts[j + 1]], fill=(6, 182, 212, int(255 * ease)), width=2)

    return img

# Definição do FPS e renderização com canal alfa real VP9 yuva420p
fps = 30
output_webm = "transparent_hud.webm"
cmd_webm = [
    "ffmpeg", "-y", "-f", "rawvideo", "-s", "1280x720", "-pix_fmt", "rgba",
    "-r", str(fps), "-i", "pipe:0", "-c:v", "libvpx-vp9", "-pix_fmt", "yuva420p",
    "-auto-alt-ref", "0", "-b:v", "0", "-crf", "28", "-deadline", "realtime", output_webm
]
proc = subprocess.Popen(cmd_webm, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
for f in range(int(fps * 3.5)):
    frame = render_transparent_hud(f / float(fps), 3.5)
    proc.stdin.write(frame.tobytes("raw", "RGBA"))
proc.stdin.close()
proc.wait()

# Exibição com fundo quadriculado xadrez para validar transparência
with open(output_webm, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")

html = f"""
<div style="display: inline-block; padding: 10px; border-radius: 8px; background: repeating-conic-gradient(#333 0% 25%, #181818 0% 50%) 50% / 20px 20px;">
  <video width="640" autoplay loop muted playsinline controls>
    <source src="data:video/webm;base64,{b64}" type="video/webm">
  </video>
</div>
"""
print("✅ Vídeo transparente gerado! Repare no fundo xadrez demonstrando o canal alfa real com fonte ampliada:")
ipd.display(ipd.HTML(html))

✅ Vídeo transparente gerado! Repare no fundo xadrez demonstrando o canal alfa real com fonte ampliada:


## 📑 4. Dossiê Investigativo: Marca-Texto Orgânico (Rough Notation)

In [10]:
def render_dossier_highlight(t: float, dur: float = 3.0, w: int = 1280, h: int = 720) -> Image.Image:
    # Background estilo papel escuro de dossiê
    img = Image.new("RGBA", (w, h), (14, 16, 20, 255))
    draw = ImageDraw.Draw(img, "RGBA")

    # Carregar fontes escaláveis
    try:
        font_title = ImageFont.truetype("LiberationSans-Bold.ttf", 24)
        font_stamp = ImageFont.truetype("LiberationSans-Bold.ttf", 20)
        font_quote = ImageFont.truetype("LiberationSans-Bold.ttf", 32)
    except IOError:
        font_title = ImageFont.load_default()
        font_stamp = ImageFont.load_default()
        font_quote = ImageFont.load_default()

    card_w, card_h = int(w * 0.85), int(h * 0.60)
    x0 = (w - card_w) // 2
    y0 = (h - card_h) // 2

    # Box principal
    draw.rounded_rectangle([x0, y0, x0 + card_w, y0 + card_h], radius=12, fill=(22, 26, 33, 255), outline=(245, 158, 11, 150), width=1)
    draw.text((x0 + 40, y0 + 40), "DOCUMENTO CONFIDENCIAL // REGISTRO OFICIAL", fill=(245, 158, 11, 245), font=font_title)

    # Carimbo "DESCLASSIFICADO"
    stamp_fade = int(min(255, max(0, (t - 0.5) * 510)))
    if stamp_fade > 0:
        draw.rectangle([x0 + card_w - 260, y0 + 34, x0 + card_w - 40, y0 + 74], outline=(239, 68, 68, stamp_fade), width=3)
        draw.text((x0 + card_w - 240, y0 + 42), "DESCLASSIFICADO", fill=(239, 68, 68, stamp_fade), font=font_stamp)

    # Texto e Marca-texto orgânico animado (Rough Notation)
    quote = "EVIDÊNCIA CONCLUSIVA: ANOMALIA ELETROMAGNÉTICA DETECTADA"
    text_y = y0 + 180

    # Tempo de animação do grifo
    p_hl = min(1.0, max(0.0, (t - 0.8) / 1.5))
    ease_hl = 1.0 - pow(1.0 - p_hl, 3) # cubic ease out

    # Largura total calculada baseada na fonte e proporção aproximada
    max_w = int(len(quote) * 19.5)
    hl_w = int(max_w * ease_hl)

    if hl_w > 10:
        steps = max(16, hl_w // 15)
        poly = []
        # Borda superior com perturbação orgânica
        for i in range(steps + 1):
            rel = i / steps
            px = x0 + 40 + int(rel * hl_w)
            py = text_y + 12 + int(np.sin(rel * 14.0) * 3.5 + np.cos(rel * 6.0) * 2.0)
            poly.append((px, py))
        # Borda inferior com perturbação orgânica dessincronizada
        for i in range(steps, -1, -1):
            rel = i / steps
            px = x0 + 40 + int(rel * hl_w)
            py = text_y + 60 + int(np.cos(rel * 12.0) * 3.0 + np.sin(rel * 5.0) * 2.5)
            poly.append((px, py))

        # Desenha o marcador amarelo
        draw.polygon(poly, fill=(245, 158, 11, 95))

    # Renderiza o texto principal por cima do marca-texto
    text_fade = int(min(255, max(0, t * 255)))
    draw.text((x0 + 44, text_y + 16), quote, fill=(248, 250, 252, text_fade), font=font_quote)

    return img

# Gerar vídeo MP4 para a Seção 4
fps = 30
dur = 3.0
output_dossier = "dossier_highlight.mp4"
cmd_dossier = [
    "ffmpeg", "-y", "-f", "rawvideo", "-s", "1280x720", "-pix_fmt", "rgba",
    "-r", str(fps), "-i", "pipe:0", "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-preset", "fast", "-crf", "18", output_dossier
]
proc = subprocess.Popen(cmd_dossier, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
for f in range(int(fps * dur)):
    frame = render_dossier_highlight(f / float(fps), dur)
    proc.stdin.write(frame.tobytes("raw", "RGBA"))
proc.stdin.close()
proc.wait()

print(f"✅ Dossiê animado com marca-texto orgânico gerado: {output_dossier}")
ipd.Video(output_dossier, embed=True, width=640)

✅ Dossiê animado com marca-texto orgânico gerado: dossier_highlight.mp4
